# PMM Dynamic Full Optimization

**3000-trial Optuna search → Top-50 stress validation → Best candidate export**

This notebook:
1. Connects to MongoDB and loads NonKYC BTC-USDT 5m candles
2. Runs 3000 Optuna trials (walk-forward, stress OFF for speed)
3. Extracts the top 50 candidates by objective score
4. Runs full stress tests on each of the top 50
5. Ranks by stress-validated robust score
6. Exports the winner as a Hummingbot YAML config
7. Generates a full markdown report

In [6]:
import sys, os, subprocess, time, logging

# Ensure pmm_lab is importable
PMM_DIR = "/quants-lab/research_notebooks/market_lab/pmm_dynamic"
if PMM_DIR not in sys.path:
    sys.path.insert(0, PMM_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PMM_DIR, "--quiet"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import optuna
import pmm_lab

# Suppress noisy Optuna logs during the big run
optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"pmm_lab {pmm_lab.__version__} | NumPy {np.__version__} | Optuna {optuna.__version__}")

MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print(f"MONGO_URI      : {'SET' if MONGO_URI else 'NOT SET'}")
print(f"OPTUNA_STORAGE : {'SET' if OPTUNA_STORAGE else 'NOT SET (using SQLite)'}")

pmm_lab 0.1.0 | NumPy 2.2.6 | Optuna 4.7.0
MONGO_URI      : SET
OPTUNA_STORAGE : SET


## 1. Load Candle Data

In [7]:
# this will delete all studies from the db.  sometimes necessary when testing new code.

#"""
import optuna

storage = OPTUNA_STORAGE
studies = optuna.study.get_all_study_names(storage=storage)
print(f"Found {len(studies)} studies:")
for name in studies:
    print(f"  {name}")

for name in studies:
    print(f"  Deleting: {name}")
    optuna.delete_study(study_name=name, storage=storage)
print("All studies deleted.")
#"""

Found 1 studies:
  mexc_APT-USDT_5m_sweep_mexc_v2
  Deleting: mexc_APT-USDT_5m_sweep_mexc_v2
All studies deleted.


In [8]:
from pmm_lab.data.mongo import MongoCandleLoader
from pmm_lab.data.candles import validate_candles
from pmm_lab.config.params import DataQuery
from pmm_lab.data.hashing import hash_candles
from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules
from pmm_lab.config.defaults import INTERVAL_SECONDS

# Configuration
CONNECTOR = "mexc"
TRADING_PAIR = "BTC-USDT"

CONNECTOR_INTERVALS = {
    "nonkyc": "5m",
    "mexc": "1m",
}

INTERVAL = CONNECTOR_INTERVALS.get(CONNECTOR, "5m")
BAR_INTERVAL_SECONDS = INTERVAL_SECONDS[INTERVAL]

print(f"Connector : {CONNECTOR}")
print(f"Pair      : {TRADING_PAIR}")
print(f"Interval  : {INTERVAL} ({BAR_INTERVAL_SECONDS}s per bar)")

STUDY_NAME = f"{CONNECTOR}_{TRADING_PAIR}_{INTERVAL}_pmm_dynamic_v3"

# Load candles
loader = MongoCandleLoader()
query = DataQuery(connector=CONNECTOR, trading_pair=TRADING_PAIR, interval=INTERVAL)
candles = loader.load_range(query)

# Validate
audit = validate_candles(candles, interval=INTERVAL, strict=True)
DATASET_HASH = hash_candles(candles)

# Load exchange rules
rules_db = load_exchange_rules()
pair_rules = resolve_pair_rules(rules_db, CONNECTOR, TRADING_PAIR)

# Reference price
ref_price = float(np.median(candles["close"]))

# Auto-scale walk-forward windows based on available data
dataset_days = len(candles) * BAR_INTERVAL_SECONDS / 86400

if dataset_days >= 120:
    TRAIN_DAYS, TEST_DAYS, STEP_DAYS = 42.0, 14.0, 14.0
elif dataset_days >= 60:
    TRAIN_DAYS, TEST_DAYS, STEP_DAYS = 21.0, 7.0, 7.0
elif dataset_days >= 28:
    TRAIN_DAYS, TEST_DAYS, STEP_DAYS = 10.0, 4.0, 4.0
else:
    raise ValueError(f"Only {dataset_days:.1f} days of data — need at least 28 for walk-forward")

print(f"Candles loaded : {len(candles):,}")
print(f"Date range     : {candles['timestamp'][0]} → {candles['timestamp'][-1]}")
print(f"Dataset days   : {dataset_days:.1f}")
print(f"Walk-forward   : train={TRAIN_DAYS}d, test={TEST_DAYS}d, step={STEP_DAYS}d")
print(f"Audit strict   : {'PASS' if audit.passed_strict else 'FAIL'}")
print(f"Dataset hash   : {DATASET_HASH[:16]}...")
print(f"Ref price      : {ref_price:,.2f} USDT")
print(f"Pair rules     : tick={pair_rules.price_tick}, step={pair_rules.amount_step}, "
      f"maker={pair_rules.fees.maker_fee}, taker={pair_rules.fees.taker_fee}")

Connector : mexc
Pair      : BTC-USDT
Interval  : 1m (60s per bar)
Candles loaded : 51,652
Date range     : 1769902200 → 1773001260
Dataset days   : 35.9
Walk-forward   : train=10.0d, test=4.0d, step=4.0d
Audit strict   : PASS
Dataset hash   : f4e8196929cb96e2...
Ref price      : 68,134.86 USDT
Pair rules     : tick=0.01, step=1e-05, maker=0.0002, taker=0.0002


## 2. Phase 1: 3000-Trial Optimization (Stress OFF)

Fast exploration with walk-forward scoring only. Stress testing is disabled
to keep each trial under 1 second. ~30 minutes total.

In [4]:
from pmm_lab.optuna.study import create_study, run_optimization
from pmm_lab.optuna.objective_wrapper import create_objective
from pmm_lab.optuna.callbacks import DegeneracyCheckCallback, TrialLoggingCallback

N_TRIALS = 3000

study = create_study(
    study_name=STUDY_NAME,
    storage_url=OPTUNA_STORAGE if OPTUNA_STORAGE else None,
    n_startup_trials=int(N_TRIALS * 0.10),  # 10% of 3000 = 300 random, then Bayesian
)

objective_fn = create_objective(
    candles=candles,
    pair_rules=pair_rules,
    bar_interval_seconds=BAR_INTERVAL_SECONDS,
    dataset_hash=DATASET_HASH,
    reference_price=ref_price,
    train_days=TRAIN_DAYS,      # NOT 42.0
    test_days=TEST_DAYS,         # NOT 14.0
    step_days=STEP_DAYS,         # NOT 14.0
    run_stress=False,
)

print(f"Starting {N_TRIALS} trials...")
t0 = time.time()

study.optimize(
    objective_fn,
    n_trials=N_TRIALS,
    callbacks=[TrialLoggingCallback(log_every=100), DegeneracyCheckCallback()],
    catch=(Exception,),
    show_progress_bar=True,
    n_jobs=8,
)

elapsed = time.time() - t0
n_complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])

print(f"\nPhase 1 complete in {elapsed/60:.1f} minutes")
print(f"  Completed: {n_complete}")
print(f"  Pruned:    {n_pruned}")
print(f"  Best:      {study.best_value:.4f} (trial {study.best_trial.number})")

Starting 3000 trials...


/quants-lab/research_notebooks/market_lab/pmm_dynamic/pmm_lab/optuna/study.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(


  0%|          | 0/3000 [00:00<?, ?it/s]


Phase 1 complete in 53.3 minutes
  Completed: 2265
  Pruned:    735
  Best:      3.9628 (trial 2523)


## 3. Extract Top 50 Candidates

In [5]:
from pmm_lab.optuna.canonicalizer import canonicalize_params

# Get all completed trials, sorted by objective
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
ranked = sorted(completed, key=lambda t: t.value, reverse=True)

TOP_N = min(50, len(ranked))
top_trials = ranked[:TOP_N]

# Reconstruct SimConfig for each
top_candidates = []
for trial in top_trials:
    config, reject = canonicalize_params(trial.params, pair_rules, ref_price)
    if config is not None:
        top_candidates.append({
            "trial_number": trial.number,
            "phase1_score": trial.value,
            "params": trial.params,
            "config": config,
        })

print(f"Top {TOP_N} trials extracted, {len(top_candidates)} valid configs reconstructed")
print(f"\nTop 10 Phase 1 scores:")
for i, c in enumerate(top_candidates[:10]):
    print(f"  #{i+1}: Trial {c['trial_number']:4d}  score={c['phase1_score']:.4f}")

Top 50 trials extracted, 50 valid configs reconstructed

Top 10 Phase 1 scores:
  #1: Trial 2523  score=3.9628
  #2: Trial 2818  score=3.7874
  #3: Trial 2859  score=3.4499
  #4: Trial 2393  score=3.0318
  #5: Trial 2536  score=2.7801
  #6: Trial 1858  score=2.7696
  #7: Trial 2611  score=2.7623
  #8: Trial 2731  score=2.7479
  #9: Trial 1786  score=2.7300
  #10: Trial 2017  score=2.6963


## 4. Phase 2: Stress Test Top 50

Run full stress tests on each candidate. This takes ~5-10 seconds per candidate,
so ~5-8 minutes total for 50 candidates.

In [6]:
from pmm_lab.objective.stress import run_stress_tests
from pmm_lab.objective.objective import ObjectiveWeights

print(f"Stress testing {len(top_candidates)} candidates...")
t0 = time.time()

for i, candidate in enumerate(top_candidates):
    config = candidate["config"]

    stress_report = run_stress_tests(
        candles=candles,
        config=config,
        pair_rules=pair_rules,
        bar_interval_seconds=BAR_INTERVAL_SECONDS,
    )

    candidate["stress_report"] = stress_report
    candidate["baseline_score"] = stress_report.baseline_objective.raw_score
    candidate["worst_scenario"] = stress_report.worst_scenario
    candidate["worst_score"] = stress_report.worst_score

    # Robust score: average of baseline and worst
    # A truly robust strategy doesn't collapse under stress
    candidate["robust_score"] = (
        0.5 * stress_report.baseline_objective.raw_score
        + 0.5 * stress_report.worst_score
    )

    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/{len(top_candidates)} done ({elapsed:.0f}s)")

elapsed = time.time() - t0
print(f"\nPhase 2 complete in {elapsed/60:.1f} minutes")

Stress testing 50 candidates...
  10/50 done (71s)
  20/50 done (141s)
  30/50 done (221s)
  40/50 done (287s)
  50/50 done (352s)

Phase 2 complete in 5.9 minutes


## 5. Rank by Stress-Validated Score

In [7]:
# Sort by robust score (higher is better)
top_candidates.sort(key=lambda c: c["robust_score"], reverse=True)

# Display top 20
rows = []
for i, c in enumerate(top_candidates[:20]):
    sr = c["stress_report"]
    bm = sr.baseline_metrics
    rows.append({
        "Rank": i + 1,
        "Trial": c["trial_number"],
        "Phase1": f"{c['phase1_score']:.2f}",
        "Baseline": f"{c['baseline_score']:.2f}",
        "Worst": f"{c['worst_score']:.2f}",
        "Robust": f"{c['robust_score']:.2f}",
        "PnL%": f"{bm.pnl_pct:.1f}",
        "Sharpe": f"{bm.sharpe:.2f}",
        "MaxDD%": f"{bm.max_drawdown_pct:.1f}",
        "Trades": bm.trade_count,
        "Fees": f"{bm.total_fees_quote:.1f}",
        "WorstScenario": c["worst_scenario"],
    })

results_df = pd.DataFrame(rows)
print("=== Top 20 Stress-Validated Candidates ===\n")
display(results_df)

=== Top 20 Stress-Validated Candidates ===



,Rank,Trial,Phase1,Baseline,Worst,Robust,PnL%,Sharpe,MaxDD%,Trades,Fees,WorstScenario
0,1,1966,2.53,1.50,-2.06,-0.28,1.1,2.30,1.6,800,2.9,extreme_slippage
1,2,2270,2.46,-0.09,-4.01,-2.05,1.5,1.12,3.5,554,3.6,extreme_slippage
2,3,2818,3.79,6.16,-13.61,-3.72,14.7,2.85,18.5,892,4.7,extreme_slippage
3,4,1486,2.46,-3.59,-4.41,-4.00,-1.4,-2.08,2.2,648,4.3,fees_2x
4,5,2768,2.57,-2.86,-5.33,-4.10,-1.0,-1.63,2.2,918,4.1,extreme_slippage
5,6,2689,2.63,-3.26,-6.10,-4.68,0.3,0.26,6.8,1134,15.6,fees_2x
6,7,2996,2.51,-3.03,-6.47,-4.75,-1.0,-1.13,2.6,684,4.9,extreme_slippage
7,8,2787,2.56,-2.50,-7.09,-4.80,-0.7,-0.73,2.6,765,2.8,extreme_slippage
8,9,2192,2.46,-3.61,-6.13,-4.87,-1.6,-4.21,2.1,665,4.3,extreme_slippage
9,10,2495,2.51,-3.51,-6.31,-4.91,-1.4,-2.32,2.7,1015,5.8,extreme_slippage


## 6. Best Candidate Deep Dive

In [8]:
from pmm_lab.objective.walkforward import run_walk_forward

best = top_candidates[0]
best_config = best["config"]
best_stress = best["stress_report"]

print("=" * 60)
print(f"BEST CANDIDATE: Trial {best['trial_number']}")
print("=" * 60)
print(f"\nRobust Score : {best['robust_score']:.4f}")
print(f"Phase1 Score : {best['phase1_score']:.4f}")
print(f"Baseline Obj : {best['baseline_score']:.4f}")
print(f"Worst Stress : {best['worst_score']:.4f} ({best['worst_scenario']})")

# Baseline metrics
bm = best_stress.baseline_metrics
print(f"\n--- Baseline Metrics ---")
print(f"  PnL %          : {bm.pnl_pct:.4f}")
print(f"  Net PnL (quote): {bm.net_pnl_quote:.4f}")
print(f"  Sharpe         : {bm.sharpe:.4f}")
print(f"  Max Drawdown % : {bm.max_drawdown_pct:.4f}")
print(f"  Profit Factor  : {bm.profit_factor:.4f}")
print(f"  Trade Count    : {bm.trade_count}")
print(f"  Total Fees     : {bm.total_fees_quote:.4f}")
print(f"  Fee Drag %     : {bm.fee_drag_pct:.4f}")

# Parameters
print(f"\n--- Parameters ---")
for k, v in sorted(best["params"].items()):
    print(f"  {k:30s} = {v}")

# Stress breakdown
print(f"\n--- Stress Scenarios ---")
print(f"  {'Scenario':<25s} {'PnL%':>8s} {'Sharpe':>8s} {'MaxDD%':>8s} {'Score':>10s}")
print(f"  {'-'*25} {'-'*8} {'-'*8} {'-'*8} {'-'*10}")
for sr in best_stress.scenario_results:
    m = sr.metrics
    print(f"  {sr.scenario.name:<25s} {m.pnl_pct:>8.2f} {m.sharpe:>8.2f} "
          f"{m.max_drawdown_pct:>8.2f} {sr.objective.raw_score:>10.4f}")

# Walk-forward on best
print(f"\n--- Walk-Forward (re-running on best config) ---")
wf_result = run_walk_forward(
    candles=candles,
    config=best_config,
    pair_rules=pair_rules,
    bar_interval_seconds=BAR_INTERVAL_SECONDS,
    dataset_hash=DATASET_HASH,
    train_days=TRAIN_DAYS,
    test_days=TEST_DAYS,
    step_days=STEP_DAYS,
)
print(f"  Aggregate score: {wf_result.aggregate_score:.4f}")
print(f"  {'Fold':>4s} {'PnL%':>8s} {'Sharpe':>8s} {'MaxDD%':>8s} {'Trades':>7s} {'Score':>10s}")
for fr in wf_result.folds:
    tm = fr.test_metrics
    print(f"  {fr.fold_index:>4d} {tm.pnl_pct:>8.2f} {tm.sharpe:>8.2f} "
          f"{tm.max_drawdown_pct:>8.2f} {tm.trade_count:>7d} {fr.test_objective.raw_score:>10.4f}")

BEST CANDIDATE: Trial 1966

Robust Score : -0.2775
Phase1 Score : 2.5282
Baseline Obj : 1.5006
Worst Stress : -2.0556 (extreme_slippage)

--- Baseline Metrics ---
  PnL %          : 1.0911
  Net PnL (quote): 7.0382
  Sharpe         : 2.2955
  Max Drawdown % : 1.6125
  Profit Factor  : 1.0650
  Trade Count    : 800
  Total Fees     : 2.9251
  Fee Drag %     : 0.4535

--- Parameters ---
  amount_skew                    = 1.6781071215712258
  buy_n_levels                   = 6
  buy_side_weight                = 0.271682898373902
  buy_spread_base                = 5.41654883336513
  buy_spread_ratio               = 2.81517229926326
  cooldown_time                  = 987.7051760923193
  executor_refresh_time          = 2527.3953982193825
  macd_fast                      = 9
  macd_signal                    = 16
  macd_slow                      = 59
  natr_length                    = 28
  sell_n_levels                  = 9
  sell_spread_base               = 0.3898801956228725
  sell_spread_r

## 7. Export YAML + Report

In [9]:
from pmm_lab.export.hb_yaml import export_yaml, ExportParams
from pmm_lab.export.validate_export import validate_yaml_file
from pmm_lab.metrics.metrics import compute_metrics
from pmm_lab.objective.objective import objective_v1
from pmm_lab.report.report_md import generate_report, run_stop_ship_checks
from pmm_lab.sim.runner import CandleSimRunner

# Export YAML
export_params = ExportParams(
    connector_name=CONNECTOR,
    trading_pair=TRADING_PAIR,
    candles_connector=CONNECTOR,
    candles_trading_pair=TRADING_PAIR,
    interval=INTERVAL,
)

yaml_path = export_yaml(
    config=best_config,
    output_path=f"artifacts/{STUDY_NAME}_best.yml",
    export_params=export_params,
    metadata={
        "dataset_hash": DATASET_HASH,
        "trial": best["trial_number"],
        "phase1_score": best["phase1_score"],
        "robust_score": best["robust_score"],
        "worst_scenario": best["worst_scenario"],
        "worst_score": best["worst_score"],
    },
)
print(f"YAML exported: {yaml_path}")

# Validate
validation = validate_yaml_file(yaml_path)
print(f"Validation: {'PASS' if validation.valid else 'FAIL'} (mode: {validation.mode})")
if not validation.valid:
    for err in validation.errors:
        print(f"  ERROR: {err}")

# Compute best metrics for report
runner = CandleSimRunner(best_config, pair_rules)
best_result = runner.run(candles)
best_metrics = compute_metrics(best_result, best_config.total_amount_quote, candles, BAR_INTERVAL_SECONDS)
best_obj = objective_v1(best_metrics)

# Stop-ship checks
checks = run_stop_ship_checks(
    best_metrics=best_metrics,
    best_objective=best_obj,
    walkforward_result=wf_result,
    stress_report=best_stress,
    dataset_hash=DATASET_HASH,
    validation_result=validation,
)

print(f"\n--- Stop-Ship Checks ---")
all_pass = True
for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  {name}: {status}")
    if not passed:
        all_pass = False

# Generate report
dataset_summary = {
    "connector": CONNECTOR,
    "trading_pair": TRADING_PAIR,
    "interval": INTERVAL,
    "n_candles": len(candles),
    "dataset_hash": DATASET_HASH,
    "n_trials_phase1": N_TRIALS,
    "n_candidates_stressed": len(top_candidates),
}

report_path = generate_report(
    study_name=STUDY_NAME,
    dataset_summary=dataset_summary,
    best_params=best["params"],
    best_metrics=best_metrics,
    best_objective=best_obj,
    walkforward_result=wf_result,
    stress_report=best_stress,
    stop_ship_checks=checks,
    output_path=f"artifacts/{STUDY_NAME}_report.md",
)
print(f"\nReport saved: {report_path}")

if all_pass:
    print("\n\u2713 ALL STOP-SHIP CHECKS PASSED \u2014 config is a candidate for paper trading")
else:
    print("\n\u2717 STOP-SHIP CHECK(S) FAILED \u2014 do NOT trade this config live")

YAML exported: artifacts/mexc_BTC-USDT_1m_pmm_dynamic_v3_best.yaml
Validation: PASS (mode: mirror)

--- Stop-Ship Checks ---
  dataset_audit: PASS
  feature_parity: PASS
  objective_not_degenerate: PASS
  stress_not_collapsed: FAIL
  yaml_validates: PASS
  determinism: PASS

Report saved: # PMM Dynamic Optimization Report: mexc_BTC-USDT_1m_pmm_dynamic_v3

Generated: 2026-03-07 05:04:44 UTC

## Dataset Summary

- **connector**: mexc
- **trading_pair**: BTC-USDT
- **interval**: 1m
- **n_candles**: 48885
- **dataset_hash**: 2b7e0ceee9978061dbdffa0c69349d9a905814410f04eee58d2ecfb177dddde0
- **n_trials_phase1**: 3000
- **n_candidates_stressed**: 50

## Best Parameters

| Parameter | Value |
|-----------|-------|
| amount_skew | 1.6781071215712258 |
| buy_n_levels | 6 |
| buy_side_weight | 0.271682898373902 |
| buy_spread_base | 5.41654883336513 |
| buy_spread_ratio | 2.81517229926326 |
| cooldown_time | 987.7051760923193 |
| executor_refresh_time | 2527.3953982193825 |
| macd_fast | 9 |
| m

## 8. Summary

Check the `artifacts/` folder for:
- `*_best.yml` — Hummingbot controller config (copy to your trading pod)
- `*_best.meta.yml` — metadata with dataset hash, trial number, scores
- `*_report.md` — full optimization report

**Before live trading:**
1. All stop-ship checks must PASS
2. Review the stress test results — worst scenario should still be profitable or near breakeven
3. Paper trade the config first using Hummingbot's paper trading mode
4. Monitor for at least 1 week before committing real capital